In [ ]:
#setup
import math
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)
print(sys.executable)
print("NumPy", np.__version__)

In [ ]:
#same symbols, different meanings
lst = [1, 2, 3]
arr = np.array([1, 2, 3])
print("list + list :", lst + lst)
print("array + array:", arr + arr)
print("list * 2    :", lst * 2)
print("array * 2   :", arr * 2)
print("array + 1   :", arr + 1)
try:
    lst + 1
except TypeError as e:
    print("TypeError for list + 1:", e)

print(np.array([1, 2.5, True]))                    # everything becomes float
print(np.array([1, "a", 2.5]))                     # everything becomes a STRING
print(np.array([1, 2, 3]).dtype, np.array([1.0, 2, 3]).dtype)
print(type(arr), type(arr[0]))                     # the elements are NumPy scalars, not Python ints

In [ ]:
#memory and speed
n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n, dtype=np.int64)

list_bytes = sys.getsizeof(py_list) + sum(sys.getsizeof(v) for v in py_list)
print(f"list : about {list_bytes / 1e6:.0f} MB")
print(f"array: {np_arr.nbytes / 1e6:.0f} MB")

def best_time(f, repeat=5):
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        f()
        best = min(best, time.perf_counter() - start)
    return best

t_loop = best_time(lambda: sum(v * v for v in py_list))
t_vec = best_time(lambda: (np_arr * np_arr).sum())
t_dot = best_time(lambda: np_arr @ np_arr)
print(f"sum of squares of {n:,} numbers")
print(f"  Python loop : {t_loop * 1000:7.1f} ms")
print(f"  NumPy       : {t_vec * 1000:7.1f} ms   ({t_loop / t_vec:.0f}x faster)")
print(f"  dot product : {t_dot * 1000:7.1f} ms   ({t_loop / t_dot:.0f}x faster)")
print("same answer:", sum(v * v for v in py_list) == int((np_arr * np_arr).sum()))

In [ ]:
#dtypes, and what overflow looks like
print(np.array([1, 2, 3]).dtype, np.array([1.5]).dtype, np.array([True]).dtype, np.array(["a"]).dtype)
print(np.iinfo(np.int8), np.iinfo(np.uint8).max, np.finfo(np.float64).eps)

a = np.array([250, 251, 252], dtype=np.uint8)               # 8-bit unsigned: only 0 to 255
print("uint8 + 10 :", a + 10)                                # wraps around, silently
print("as int16   :", a.astype(np.int16) + 10)
b = np.array([127], dtype=np.int8)
print("int8 127 + 1:", b + 1)                                # -128

print(np.array([7, 2]) / 2, np.array([7, 2]) // 2)           # / is always float, // floors
x = np.array([1, 2, 3])
x[0] = 9.7                                                    # assigning a float into an int array TRUNCATES
print(x)
y = np.array([1.9, -1.9]).astype(int)
print(y, np.array([1.5, 2.5, 3.5]).round())                  # astype truncates; round() rounds half to even
print(0.1 + 0.2 == 0.3, np.isclose(0.1 + 0.2, 0.3), np.allclose([0.1 + 0.2], [0.3]))

In [ ]:
#ways to create arrays
print(np.array([[1, 2, 3], [4, 5, 6]]))                      # from nested lists
print(np.zeros((2, 3)), np.ones(3), np.full((2, 2), 7), sep="\n")
print(np.eye(3))                                              # the identity matrix
print(np.arange(0, 10, 2), np.arange(5))                      # like range(), but returns an array
print(np.linspace(0, 1, 5))                                   # 5 evenly spaced points, endpoint INCLUDED
print(np.logspace(0, 3, 4))                                   # 1, 10, 100, 1000
print(rng.integers(1, 7, size=(2, 5)))                        # dice: 1 to 6, the upper bound is excluded
print(rng.normal(0, 1, size=3).round(2), rng.random(3).round(2))
print(np.zeros_like(np.arange(4.0)), np.diag([1, 2, 3]), np.tile([1, 2], 3), np.repeat([1, 2], 3))

In [ ]:
#shape, dimensions, and reshaping
m = np.arange(12).reshape(3, 4)
print(m)
print("shape", m.shape, "| ndim", m.ndim, "| size", m.size, "| dtype", m.dtype,
      "| itemsize", m.itemsize, "| nbytes", m.nbytes)

print(m.reshape(2, 6).shape, m.reshape(2, -1).shape, m.reshape(-1).shape)     # -1 means "work it out"
print(m.T.shape, np.arange(3).reshape(3, 1).shape)

try:
    m.reshape(5, 3)
except ValueError as e:
    print("ValueError:", e)

v = np.array([1, 2, 3])
print("1-D shape:", v.shape, "| as a column:", v[:, None].shape, "| as a row:", v[None, :].shape)
cube = np.arange(24).reshape(2, 3, 4)                        # 2 blocks, 3 rows, 4 columns
print(cube.shape, cube[1, 2, 3])

In [ ]:
#vector arithmetic and functions
a = np.array([3, 4, 6])
b = np.array([1, 2, 3])
print(a + b, a - b, a * b, a / b, a ** 2, a % 2, -a)
print(a > b, a == b, (a > 3).sum())                          # comparisons give boolean arrays
print(np.sqrt(a), np.exp([0, 1]), np.log([1, np.e]), np.abs([-2, 3]))
print(np.round([1.234, 5.678], 1), np.floor([1.7, -1.7]), np.ceil([1.2, -1.2]))
print(np.maximum(a, b), np.minimum(a, 4), np.clip(a, 2, 5))
print(np.sqrt(-1.0))                                          # nan, with a RuntimeWarning

In [ ]:
#dot product, length, angle, projection
u = np.array([3, 4])
v = np.array([1, 2])
print("dot product:", np.dot(u, v), u @ v, (u * v).sum())              # 3*1 + 4*2 = 11
print("length |u| :", np.linalg.norm(u))                               # 5.0
unit = u / np.linalg.norm(u)
print("unit vector:", unit, np.linalg.norm(unit))
cosine = u @ v / (np.linalg.norm(u) * np.linalg.norm(v))
print(f"cosine similarity {cosine:.4f}, angle {np.degrees(np.arccos(cosine)):.2f} degrees")
print("projection of v on u:", (v @ u) / (u @ u) * u)
print("distance between the points:", np.linalg.norm(u - v))

print("perpendicular vectors:", np.array([1, 0]) @ np.array([0, 5]))    # dot = 0

In [ ]:
#aggregation functions
x = np.array([4, 8, 6, 5, 3, 8, 9, 7, 8, 5])                  # the data from S1
print("sum", x.sum(), "| mean", x.mean(), "| median", np.median(x), "| min", x.min(), "| max", x.max())
print("range", np.ptp(x), "| product of first 4", x[:4].prod(), "| cumulative sum", x.cumsum())
print("argmin", x.argmin(), "(value", x[x.argmin()], ") | argmax", x.argmax(), "(value", x[x.argmax()], ")")
print("population SD", x.std().round(4), "| sample SD", x.std(ddof=1).round(4), "| sample variance", x.var(ddof=1).round(4))
print("percentiles:", np.percentile(x, [25, 50, 75]))
values, counts = np.unique(x, return_counts=True)
print("value counts:", dict(zip(values.tolist(), counts.tolist())), "| mode:", values[counts.argmax()])

In [ ]:
#statistics as vector operations
marks, weights = np.array([80, 90, 70]), np.array([0.3, 0.5, 0.2])
print("weighted mean = dot product:", marks @ weights, np.average(marks, weights=weights))

z = (x - x.mean()) / x.std(ddof=1)                             # z-scores for the whole array at once
print("z-scores:", z.round(2))
print("mean of z", z.mean().round(10), "| SD of z", z.std(ddof=1).round(4))

q1, q3 = np.percentile(x, [25, 75])
iqr = q3 - q1
outliers = x[(x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)]
print("IQR", iqr, "| outliers:", outliers)

deviations = x - x.mean()
print("sum of squared deviations", (deviations ** 2).sum().round(2), "| variance", ((deviations ** 2).sum() / (len(x) - 1)).round(4))

In [ ]:
#Matrices

#A matrix is a 2-D array (rows × columns).

#* vs @

A = np.array([[1, 2], [3, 4]])
B = np.array([[0, 1], [1, 0]])
print("A * B (element-wise):\n", A * B)
print("A @ B (matrix product):\n", A @ B)
print("B @ A:\n", B @ A)
print("matrix multiplication is NOT commutative:", np.array_equal(A @ B, B @ A))
print("transpose:\n", A.T)
print("identity leaves A unchanged:", np.array_equal(A @ np.eye(2), A))
print("diagonal", np.diag(A), "| trace", np.trace(A))

M = np.arange(6).reshape(2, 3)                                 # (2, 3)
N = np.arange(6).reshape(3, 2)                                 # (3, 2)
print("(2,3) @ (3,2) gives", (M @ N).shape, "| (3,2) @ (2,3) gives", (N @ M).shape)
try:
    M @ M
except ValueError as e:
    print("ValueError:", e)
print("matrix times vector:", A @ np.array([1, 1]))

In [ ]:
#determinant, inverse, and solving equations
A = np.array([[1.0, 2.0], [3.0, 4.0]])
print("determinant:", round(np.linalg.det(A), 6))               # 1*4 - 2*3 = -2
Ainv = np.linalg.inv(A)
print("inverse:\n", Ainv)
print("A @ inverse = identity:", np.allclose(A @ Ainv, np.eye(2)))

# 2x + y = 5 and x + 3y = 10
coef = np.array([[2.0, 1.0], [1.0, 3.0]])
rhs = np.array([5.0, 10.0])
sol = np.linalg.solve(coef, rhs)
print("solution (x, y):", sol, "| check:", coef @ sol)

singular = np.array([[1.0, 2.0], [2.0, 4.0]])                   # the second row is twice the first
print("rank:", np.linalg.matrix_rank(singular), "| determinant:", np.linalg.det(singular))
try:
    np.linalg.inv(singular)
except np.linalg.LinAlgError as e:
    print("LinAlgError:", e)

In [ ]:
#he covariance matrix from S7, built with a matrix product
X = rng.multivariate_normal([0, 0, 0], [[1, 0.8, 0.2], [0.8, 1, 0.3], [0.2, 0.3, 1]], size=500)
Xc = X - X.mean(axis=0)                                         # centre each column
cov_manual = Xc.T @ Xc / (len(X) - 1)
print(np.round(cov_manual, 3))
print("matches np.cov:", np.allclose(cov_manual, np.cov(X, rowvar=False)))
print("np.cov(X).shape (default rowvar=True):", np.cov(X).shape, "<- it treated ROWS as the variables")

sd = np.sqrt(np.diag(cov_manual))
corr = cov_manual / np.outer(sd, sd)                            # correlation = covariance / (sd_i * sd_j)
print("correlation matrix matches:", np.allclose(corr, np.corrcoef(X, rowvar=False)))

eigenvalues, eigenvectors = np.linalg.eigh(cov_manual)          # eigh: for symmetric matrices
print("eigenvalues (ascending):", eigenvalues.round(3))
print("share of the variance on each principal axis:", (eigenvalues[::-1] / eigenvalues.sum()).round(3))

In [ ]:
#Indexing and slicing
# 1-D arrays
a = np.array([10, 20, 30, 40, 50, 60])
#   index:      0   1   2   3   4   5
#   negative:  -6  -5  -4  -3  -2  -1
print(a[0], a[-1], a[1:4], a[:3], a[3:], a[::2], a[::-1], a[-3:])
print("fancy index with a list:", a[[0, 2, 5]], "| repeated:", a[[1, 1, 1]])
print("out of range slice is fine:", a[4:100], "| empty slice:", a[3:3])
try:
    a[6]
except IndexError as e:
    print("IndexError:", e)

a[1:3] = 0                                                      # assign to a slice
print(a)
a[:] = 5
print(a)

In [ ]:
#2-D arrays, a[row, column]

m = np.arange(1, 17).reshape(4, 4)
print(m)
print("m[1, 2]      :", m[1, 2], "| m[1][2] gives the same here:", m[1][2], "(but builds an extra array)")
print("row 0        :", m[0], "| last row:", m[-1])
print("column 1     :", m[:, 1], "| last column:", m[:, -1])
print("block\n", m[1:3, 1:3])
print("every 2nd row and column\n", m[::2, ::2])
print("rows 0 and 3, columns 0 and 3\n", m[np.ix_([0, 3], [0, 3])])
print("pairs (0,0) and (3,3):", m[[0, 3], [0, 3]], "<- two lists pair up, they don't form a grid")
print("diagonal:", np.diag(m), "| upper triangle:", m[np.triu_indices(4)])

m[:, 0] = 0                                                      # set a whole column
m[0, :] = -1                                                     # set a whole row
print(m)

cube = np.arange(24).reshape(2, 3, 4)
print(cube[1], "\n", cube[:, 0, :], "\n", cube[..., 0])           # ... means "all remaining axes"

In [ ]:
#views vs copies
base = np.arange(6)
view = base[1:4]                                                 # a slice is a VIEW: same memory
view[0] = 99
print("base after changing the view:", base)

fancy = base[[1, 2, 3]]                                          # fancy indexing makes a COPY
fancy[0] = -1
print("base after changing the fancy copy:", base)
print("shares memory:", np.shares_memory(base, view), np.shares_memory(base, fancy))

safe = base[1:4].copy()                                          # make the copy explicit
safe[0] = 0
print("base unchanged:", base)

alias = base                                                      # plain assignment: the same object
alias[0] = 1234
print("alias:", base[0], "| copy:", base.copy() is base)
print("reshape and ravel return views when they can:", np.shares_memory(base, base.reshape(2, 3)), np.shares_memory(base, base.ravel()),
      "| flatten always copies:", np.shares_memory(base, base.flatten()))

In [ ]:
#boolean masks and where
scores = np.array([55, 72, 38, 91, 66, 45, 80, 29])
mask = scores >= 50
print(mask, "| passed:", scores[mask], "| count:", mask.sum(), "| proportion:", mask.mean())
print("between 40 and 80:", scores[(scores > 40) & (scores < 80)])
print("outside 40 to 80 :", scores[(scores <= 40) | (scores >= 80)])
print("not passed       :", scores[~mask])
try:
    scores[scores > 40 and scores < 80]
except ValueError as e:
    print("ValueError:", e)

print("any above 90:", (scores > 90).any(), "| all above 20:", (scores > 20).all())
print("grades:", np.where(scores >= 50, "pass", "fail"))
print("replace the failures with 50:", np.where(scores < 50, 50, scores))
print("indices of the passes:", np.flatnonzero(mask), "| np.where(mask)[0]:", np.where(mask)[0])

capped = scores.copy()
capped[capped > 85] = 85                                          # assign through a mask
print("capped:", capped)

order = np.argsort(scores)
print("ascending order of indices:", order, "| top 3 scores:", scores[order[::-1][:3]])
print("labels:", np.select([scores >= 80, scores >= 50], ["A", "B"], default="C"))
print("2-D mask:\n", np.arange(9).reshape(3, 3)[np.arange(9).reshape(3, 3) % 2 == 0])

In [ ]:
#Broadcasting
m = np.arange(12).reshape(3, 4)                                   # shape (3, 4)
row = np.array([10, 20, 30, 40])                                  # shape (4,)  -> added to every row
col = np.array([[100], [200], [300]])                             # shape (3, 1) -> added to every column
print(m + row, m + col, sep="\n\n")
print("(3, 1) + (4,) gives shape", (col + row).shape, "\n", col + row)    # an "outer sum"

try:
    m + np.array([1, 2, 3])                                        # (3, 4) + (3,): 4 vs 3 do not match
except ValueError as e:
    print("ValueError:", e)
print("fix with a column:\n", m + np.array([1, 2, 3])[:, None])

def broadcast_shape(*shapes):
    try:
        return np.broadcast_shapes(*shapes)
    except ValueError:
        return "incompatible"

for shapes in (((3, 4), (4,)), ((3, 4), (3,)), ((3, 1), (1, 4)), ((2, 3, 4), (3, 1)), ((5,), (5, 1)), ((2, 3), (2, 3, 1))):
    print(f"{str(shapes[0]):<10} + {str(shapes[1]):<10} -> {broadcast_shape(*shapes)}")

In [ ]:
#broadcasting at work in statistics
data = rng.normal([50, 170, 65], [10, 8, 12], size=(200, 3))     # 200 people, 3 measurements
Z = (data - data.mean(axis=0)) / data.std(axis=0, ddof=1)         # (200,3) - (3,) : standardise every column
print("column means", Z.mean(axis=0).round(10), "| column SDs", Z.std(axis=0, ddof=1).round(4))

M = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float)
wrong = M / M.sum(axis=1)                                          # (3,3) / (3,): divides COLUMNS, no error!
right = M / M.sum(axis=1, keepdims=True)                           # (3,3) / (3,1): divides ROWS
print("row sums, wrong:", wrong.sum(axis=1).round(3), "| right:", right.sum(axis=1))

P = np.array([[0, 0], [3, 4], [6, 8], [0, 5], [5, 0]])             # 5 points in the plane
diff = P[:, None, :] - P[None, :, :]                               # (5,1,2) - (1,5,2) -> (5,5,2)
D = np.sqrt((diff ** 2).sum(axis=-1))                              # pairwise distances, with no loops
print("distance matrix:\n", D.round(2))
print("symmetric:", np.allclose(D, D.T), "| zero diagonal:", np.allclose(np.diag(D), 0))

heights, weights = np.arange(150, 191, 10), np.array([50, 60, 70, 80])
print("BMI table (rows = height, columns = weight):\n", (weights[None, :] / (heights[:, None] / 100) ** 2).round(1))

In [ ]:
#Reshaping, combining, and housekeeping
#reshape, combine, sort, unique
a = np.arange(6)
b = np.arange(6, 12)
print(np.concatenate([a, b]), np.vstack([a, b]).shape, np.hstack([a, b]).shape, np.stack([a, b], axis=1).shape)
print(np.split(np.arange(10), [3, 7]))
grid = np.arange(12).reshape(3, 4)
print("append a column:\n", np.hstack([grid, np.zeros((3, 1), dtype=int)]))
print("ravel", grid.ravel()[:5], "| transpose shape", grid.T.shape, "| flip rows\n", np.flipud(grid)[0])

data = np.array([3, 1, 2, 3, 3, 1, 5])
print("sorted", np.sort(data), "| descending", np.sort(data)[::-1], "| argsort", np.argsort(data))
print("unique", np.unique(data), "| with counts", np.unique(data, return_counts=True))
print("isin:", np.isin(data, [1, 5]), "| set ops:", np.intersect1d([1, 2, 3], [2, 3, 4]), np.setdiff1d([1, 2, 3], [2]))
table = np.array([[3, 9], [1, 5], [2, 7]])
print("sort the rows by the first column:\n", table[np.argsort(table[:, 0])])

heights = np.array([150, 162, 171, 180, 195])
bins = np.array([160, 175, 190])
print("digitize:", np.digitize(heights, bins))                       # which bin each value falls in
counts, edges = np.histogram(rng.normal(0, 1, 1000), bins=5)
print("histogram counts:", counts, "| edges", edges.round(2))
print("bincount:", np.bincount([0, 1, 1, 3, 3, 3]))

In [ ]:
# missing values
d = np.array([4.0, np.nan, 6.0, 8.0])
print("mean:", d.mean(), "| nanmean:", np.nanmean(d), "| nanstd:", np.nanstd(d, ddof=1).round(3))
print("isnan:", np.isnan(d), "| clean:", d[~np.isnan(d)], "| nan == nan:", np.nan == np.nan)
print("fill with the mean:", np.where(np.isnan(d), np.nanmean(d), d))
print("fill with zero:", np.nan_to_num(d, nan=0.0))
print("max:", d.max(), "| nanmax:", np.nanmax(d))
try:
    np.array([1, 2, np.nan], dtype=int)
except ValueError as e:
    print("ValueError:", e)

In [ ]:
#vectorised simulation vs a Python loop
def loop_version(reps=5000, n=30):
    return np.array([np.mean([rng.exponential(5) for _ in range(n)]) for _ in range(reps)])

def vector_version(reps=5000, n=30):
    return rng.exponential(5, size=(reps, n)).mean(axis=1)

t_loop = best_time(loop_version, repeat=2)
t_vec = best_time(vector_version, repeat=5)
print(f"loop: {t_loop:.3f} s | vectorised: {t_vec * 1000:.1f} ms | {t_loop / t_vec:.0f}x faster")
print("same kind of answer:", loop_version().mean().round(2), vector_version().mean().round(2))

# Pairwise distances: nested loops vs broadcasting
pts = rng.random((400, 2))
def dist_loops():
    return np.array([[math.dist(p, q) for q in pts] for p in pts])
def dist_broadcast():
    return np.sqrt(((pts[:, None, :] - pts[None, :, :]) ** 2).sum(-1))
print(f"distance matrix, 400 points: loops {best_time(dist_loops, 2):.3f} s, broadcasting {best_time(dist_broadcast) * 1000:.1f} ms,",
      "same result:", np.allclose(dist_loops(), dist_broadcast()))

In [ ]:
#a marks matrix
# 100 students, 5 subjects, no loops
subjects = ["Maths", "Physics", "Chemistry", "English", "CS"]
ability = rng.normal(0, 8, size=(100, 1))                              # each student's general ability
difficulty = np.array([-5, -8, -3, 2, 0])                              # a per-subject effect
marks = np.clip(60 + ability + difficulty + rng.normal(0, 7, size=(100, 5)), 0, 100).round(1)
print("shape", marks.shape, "| the shapes (100,1) + (5,) + (100,5) broadcast to (100,5)")
print(marks[:3])

print("\nmeans:", dict(zip(subjects, marks.mean(axis=0).round(1).tolist())))
print("SDs  :", dict(zip(subjects, marks.std(axis=0, ddof=1).round(1).tolist())))
print("hardest subject by mean:", subjects[marks.mean(axis=0).argmin()])

fails = marks < 40
print("failures per subject:", dict(zip(subjects, fails.sum(axis=0).tolist())))
print("students failing 2 or more subjects:", (fails.sum(axis=1) >= 2).sum())

total = marks.sum(axis=1)
rank = (-total).argsort().argsort() + 1                                 # rank 1 = the highest total
top5 = np.argsort(total)[::-1][:5]
print("top 5 (index, total):", list(zip(top5.tolist(), total[top5].round(1).tolist())))
print("student", top5[0], "has rank", rank[top5[0]], "| rank of the last student in the list:", rank.max())

z = (marks - marks.mean(axis=0)) / marks.std(axis=0, ddof=1)
print("student 0, best subject (by z-score):", subjects[z[0].argmax()], "| z-scores:", z[0].round(2))
print("above the average in every subject:", (marks > marks.mean(axis=0)).all(axis=1).sum())
print("curved marks (+5, capped at 100), mean change:", (np.minimum(marks + 5, 100) - marks).mean().round(2))

print("\ncorrelation between subjects (the shared 'ability' makes them positive):")
print(np.corrcoef(marks, rowvar=False).round(2))

In [ ]:
#grayscale image, slicing, and the uint8 trap
H, W = 120, 200
y, x = np.mgrid[0:H, 0:W]                                                 # two (H, W) coordinate arrays
img = (255 * x / (W - 1)).astype(np.uint8)                                # a left-to-right gradient
inside = (x - 100) ** 2 + (y - 60) ** 2 < 40 ** 2                         # a boolean mask: a circle
img[inside] = 255 - img[inside]                                           # invert inside the circle
print(img.shape, img.dtype, img.min(), img.max(), "| pixel (60, 100):", img[60, 100], "| mean", img.mean().round(1))

flipped = img[:, ::-1]                                                    # a mirror image, by slicing
crop = img[20:100, 50:150]                                                # a crop, by slicing
bright_wrong = img + 60                                                   # wraps around above 195
bright_right = np.clip(img.astype(np.int16) + 60, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 5, figsize=(16, 2.8))
for ax, (title, im) in zip(axes, (("original", img), ("mirrored", flipped), ("crop", crop),
                                  ("+60 wrongly (wrap-around)", bright_wrong), ("+60 with clipping", bright_right))):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
#a colour image is a 3-D array
rgb = np.stack([img, img[::-1], img[:, ::-1]], axis=-1)                   # shape (H, W, 3): red, green, blue
print(rgb.shape, rgb.dtype, "| per-channel means:", rgb.mean(axis=(0, 1)).round(1))
gray = (rgb @ np.array([0.299, 0.587, 0.114])).round().astype(np.uint8)   # (H, W, 3) @ (3,) -> (H, W)

fig, axes = plt.subplots(1, 4, figsize=(14, 2.8))
axes[0].imshow(rgb)
axes[0].set_title("RGB")
axes[1].imshow(rgb[..., [2, 1, 0]])
axes[1].set_title("channels reversed")
axes[2].imshow(gray, cmap="gray")
axes[2].set_title("grayscale (a matrix product)")
axes[3].hist(rgb[..., 0].ravel(), bins=32, color="tomato")
axes[3].set_title("red-channel histogram")
for ax in axes[:3]:
    ax.axis("off")
plt.tight_layout()
plt.show()